## Exercise 1: Access Control

You are a data engineer at **NorthMart Retail**, a national supermarket chain. Your team stores customer and transaction data in Unity Catalog. Before any analyst can run queries, you must first create the catalog, the schema, and populate a customer table — then grant the `retail-analysts` group the permissions they need.

> ⚠️ Before starting this exercise, ensure you have created the `retail-analysts` group in Databricks and added your own user account to it. See the lab setup instructions.

### Task 1.1 — Create the catalog and schema

Create a catalog called `retail_catalog` with a suitable comment describing its purpose. Then create a schema called `security_lab` inside it.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a Unity Catalog catalog and schema using SQL in Azure Databricks?"*

**Hint:** Use `CREATE CATALOG IF NOT EXISTS` and `CREATE SCHEMA IF NOT EXISTS catalog.schema`.

In [ ]:
%sql
-- TODO: Create retail_catalog with a comment

-- TODO: Create the security_lab schema inside retail_catalog with a comment

### Task 1.2 — Create and populate the customers table

Create a managed Delta table `retail_catalog.security_lab.customers` using `CREATE TABLE ... AS SELECT ... FROM VALUES`. The table must contain the following columns and sample rows:

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a Delta table in Unity Catalog using CREATE TABLE AS SELECT FROM VALUES with column aliases?"*

**Hint:** Use the pattern `CREATE TABLE ... AS SELECT * FROM VALUES (...), (...) AS t(col1, col2, ...)`.

In [ ]:
%sql
-- TODO: Create retail_catalog.security_lab.customers and populate it with the 6 sample rows above

### Task 1.3 — Grant permissions to the retail-analysts group

The `retail-analysts` group needs to query the `customers` table. Grant the necessary privileges at all required levels of the Unity Catalog hierarchy:

- `USE CATALOG` on `retail_catalog`
- `USE SCHEMA` on `retail_catalog.security_lab`
- `SELECT` on `retail_catalog.security_lab.customers`

> 🤖 **Databricks Assistant tip:** Ask:
> *"What SQL grants are needed to allow a Databricks group to SELECT from a Unity Catalog table?"*

**Hint:** Use `GRANT <privilege> ON <object type> <object name> TO \`retail-analysts\``.

In [ ]:
%sql
-- TODO: Grant USE CATALOG on retail_catalog to retail-analysts

-- TODO: Grant USE SCHEMA on retail_catalog.security_lab to retail-analysts

-- TODO: Grant SELECT on retail_catalog.security_lab.customers to retail-analysts

### Task 1.4 — Verify the granted permissions

Verify that the privileges were correctly assigned. Use `SHOW GRANTS` to display all privileges on the `customers` table.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I use SHOW GRANTS to verify permissions on a Unity Catalog table?"*

**Expected result:** You should see `retail-analysts` listed with the `SELECT` privilege.

In [ ]:
%sql
-- TODO: Show all grants on retail_catalog.security_lab.customers

## Exercise 2: Row Filtering

NorthMart Retail's regional analysts should only see customer records for their own region. You will implement a **row filter function** that restricts rows based on who is querying the table.

Since you cannot create multiple user accounts, you will write a filter that shows only `North` region rows to everyone *except* for your own user account, which sees all rows. This simulates what a regional analyst would experience.

### Task 2.1 — Create a row filter function

Create a SQL function `retail_catalog.security_lab.region_filter` that takes a `region` parameter of type `STRING` and returns `TRUE` if:
- the `region` is `'North'`, **OR**
- the current user is your own account (replace `your@email.com` with your actual email).

This means: unless you are the privileged user, only `North` region rows will be visible.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a row filter function in Databricks Unity Catalog that uses current_user() to control access?"*

**Hint:** Use `CREATE OR REPLACE FUNCTION ... RETURN <boolean expression using current_user()>`.

In [ ]:
%sql
-- TODO: Create the region_filter function in retail_catalog.security_lab
-- Replace 'your@email.com' with your actual Databricks user email

### Task 2.2 — Apply the row filter to the customers table

Attach the `region_filter` function to the `customers` table so that it is automatically applied to every query against that table.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I set a row filter on a Unity Catalog table in Azure Databricks?"*

**Hint:** Use `ALTER TABLE ... SET ROW FILTER <function> ON (<column>)`.

In [ ]:
%sql
-- TODO: Apply region_filter to retail_catalog.security_lab.customers on the region column

### Task 2.3 — Test the row filter

Query the `customers` table. Because your own email is listed as the privileged user in the filter function, you should see **all 6 rows**.

If you had logged in as someone else (not your email), you would only see the 2 `North` region rows.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How can I verify which user is currently running a query in Databricks using SQL?"*

In [ ]:
%sql
-- Verify the current user
SELECT current_user();

-- TODO: Query retail_catalog.security_lab.customers and observe the results

### Task 2.4 — Remove the row filter

Now remove the row filter from the `customers` table. After removing it, query the table again to confirm **all 6 rows** are now returned regardless of user context.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I remove a row filter from a Unity Catalog table?"*

**Hint:** Use `ALTER TABLE ... DROP ROW FILTER`.

In [ ]:
%sql
-- TODO: Remove the row filter from retail_catalog.security_lab.customers

-- TODO: Query the table to confirm all 6 rows are visible

## Exercise 3: Column Masking

Customer email addresses are PII and must be masked for users who aren't privileged. You will create a **column mask function** that hides the full email address for most users, applying it to the `email` column of the `customers` table.

### Task 3.1 — Create a column mask function

Create a SQL function `retail_catalog.security_lab.mask_email` that takes an `email` parameter of type `STRING` and:
- Returns the **full email** if the current user is your own account.
- Returns a **masked value** (e.g. `'***@***.com'`) for all other users.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a column masking function in Databricks Unity Catalog that uses current_user() to decide whether to return the real or masked value?"*

**Hint:** Use a `CASE WHEN current_user() = '...' THEN email ELSE '***@***.com' END` pattern inside the function body.

In [ ]:
%sql
-- TODO: Create the mask_email function in retail_catalog.security_lab
-- Replace 'your@email.com' with your actual Databricks user email

### Task 3.2 — Apply the column mask to the email column

Apply the `mask_email` function as a column mask on the `email` column of the `customers` table.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I set a column mask on a Unity Catalog table column in Azure Databricks using SQL?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN email SET MASK <function>`.

In [ ]:
%sql
-- TODO: Apply mask_email to the email column of retail_catalog.security_lab.customers

### Task 3.3 — Test the column mask

Query the `customers` table. Since you are the privileged user specified in the mask function, you should see the **real email addresses**.

Any other user querying this table would see `***@***.com` in the `email` column.

In [ ]:
%sql
-- TODO: Query retail_catalog.security_lab.customers and observe the email column

### Task 3.4 — Remove the column mask

Remove the column mask from the `email` column, then query the table again to confirm emails are now fully visible to all users.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I drop a column mask from a Unity Catalog table column?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN email DROP MASK`.

In [ ]:
%sql
-- TODO: Remove the column mask from the email column

-- TODO: Query the table to confirm email addresses are now unmasked

## Exercise 4: Azure Key Vault Secrets

NorthMart Retail's loyalty platform requires an API key to connect. Storing this key in plain text inside a notebook is a security risk. In this exercise you retrieve the API key securely from the Azure Key Vault-backed secret scope you created during lab setup.

> ⚠️ Before running the cells in this exercise, ensure you have completed the Key Vault setup steps in the lab instructions (create Key Vault, add secret `loyalty-api-key`, create secret scope `retail-kv-scope`).

### Task 4.1 — List available secrets in the scope

List all secrets available in the `retail-kv-scope` secret scope to confirm that `loyalty-api-key` is accessible.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I list the secrets in a Databricks secret scope using dbutils?"*

**Hint:** Use `dbutils.secrets.list("scope-name")`. This returns key names only — never secret values.

In [ ]:
# TODO: List all secrets in the retail-kv-scope and display the output

### Task 4.2 — Retrieve the API key securely in Python

Retrieve the `loyalty-api-key` secret from the `retail-kv-scope` scope and assign it to a variable. Then print it.

Observe how Azure Databricks **automatically redacts** the value in the notebook output — you will see `[REDACTED]` instead of the actual key.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I retrieve a secret from a Databricks secret scope using dbutils.secrets.get and why is the output redacted?"*

In [ ]:
# TODO: Retrieve the loyalty-api-key from retail-kv-scope
# TODO: Print the api_key variable and observe that the output is [REDACTED]

### Task 4.3 — Use the secret to simulate a connection string

Construct a connection string that embeds the API key, simulating how it would be used to connect to NorthMart's loyalty platform. Print the connection string and observe that the secret value is still redacted.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I safely use a Databricks secret inside an f-string for a connection string without exposing the value?"*

In [ ]:
# TODO: Build a connection string using the api_key variable
# Example format: "https://loyalty.northmart.com/api?key=<api_key>"
# TODO: Print the connection string

### Task 4.4 — Retrieve the secret using SQL

Retrieve the same secret using the `secret()` SQL function. This is useful when constructing SQL-based connections.

Note that the output is also redacted in SQL results.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I use the secret() function in Databricks SQL to retrieve a secret from a scope?"*

In [ ]:
%sql
-- TODO: Use the secret() SQL function to retrieve loyalty-api-key from retail-kv-scope

## Cleanup

Run the cell below to remove all objects created during this lab.

In [ ]:
%sql
-- Remove all lab objects
ALTER TABLE IF EXISTS retail_catalog.security_lab.customers DROP ROW FILTER;
ALTER TABLE IF EXISTS retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS retail_catalog.security_lab.customers;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.region_filter;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.mask_email;
DROP SCHEMA IF EXISTS retail_catalog.security_lab;
DROP CATALOG IF EXISTS retail_catalog;